# Decode and budget a chat request

Run this notebook in local Jupyter or Colab. The required tasks use NumPy and fixed inputs. A small
synthetic vocabulary lets you inspect every probability. Finish by comparing three decoding settings
and writing down which memory bottleneck limits the reference workload.


In [ ]:
import numpy as np

messages = [("system", "Answer briefly."), ("user", "Capital of France?")]
vocabulary = ["Paris", "Lyon", "Rome", "banana"]
logits = np.array([3.0, 1.5, 0.5, -2.0])


## 1. Render the prompt

Concatenate one block per message in the exact form `<|im_start|>ROLE\nCONTENT<|im_end|>\n`,
then append `<|im_start|>assistant\n` to start the reply. The delimiter strings stand in for
special tokens and the newlines are literal line breaks.


In [ ]:
def render_chat(messages):
    # TODO: format each message and append the generation prompt.
    raise NotImplementedError


In [ ]:
assert render_chat(messages) == "<|im_start|>system\nAnswer briefly.<|im_end|>\n<|im_start|>user\nCapital of France?<|im_end|>\n<|im_start|>assistant\n"
assert render_chat([("user", "Hi")]) == "<|im_start|>user\nHi<|im_end|>\n<|im_start|>assistant\n"
print("OK")


## 2. Convert logits into a sampling distribution

For a strictly positive temperature, divide logits by temperature, subtract the maximum before
exponentiating, then normalise. Sort probabilities from largest to smallest using stable ties.
Keep the smallest prefix whose cumulative mass reaches top-p, including the crossing token,
and renormalise the kept probabilities. Accept `0 < p <= 1` and return probabilities in vocabulary order.


In [ ]:
def sampling_probs(logits, temperature, p):
    # TODO: stable softmax, nucleus selection and renormalisation.
    raise NotImplementedError


In [ ]:
assert np.allclose(sampling_probs(logits, 1, 0.7), [1, 0, 0, 0])
assert np.allclose(sampling_probs(np.log([0.25, 0.5, 0.25]), 1, 0.75), [1/3, 2/3, 0])
assert np.all(sampling_probs(logits + 1000, 1, 1) > 0)
rng = np.random.default_rng(20)
print("greedy", vocabulary[int(np.argmax(logits))])
for temperature, p in [(0.7, 0.95), (1.5, 1)]:
    probs = sampling_probs(logits, temperature, p)
    draws = rng.choice(vocabulary, size=500, p=probs)
    print(temperature, p, dict(zip(vocabulary, np.round(probs, 3))),
          "tail draws", int(np.sum(draws == "banana")))
print("OK")


## 3. Budget weights and context

Return total decimal GB for weights, KV cache, and fixed runtime overhead. Use effective bits per
weight. The cache holds both keys and values for every token, layer and KV head. Assume batch size one.

Compare 4,096 and 16,384 tokens for a 3B model with 5 effective bits, 28 layers, 8 KV heads of width
128, two bytes per cache entry, and 0.5 GB of fixed overhead. Which fits a 4 GB budget?


In [ ]:
def memory_gb(params, bits, tokens, layers, kv_heads, head_dim, cache_bytes, overhead_gb):
    # TODO: add weight and cache bytes, convert to decimal GB, then add overhead.
    raise NotImplementedError


In [ ]:
short = memory_gb(3e9, 5, 4096, 28, 8, 128, 2, 0.5)
long = memory_gb(3e9, 5, 16384, 28, 8, 128, 2, 0.5)
assert np.isclose(short, 2.844762048)
assert short < 4 < long
assert memory_gb(8e9, 4, 0, 28, 8, 128, 2, 0) == 4
print(f"4,096 tokens: {short:.2f} GB. 16,384 tokens: {long:.2f} GB.")
print("OK")


## 4. Abstain on weak agreement

Normalise short answers by stripping whitespace and folding case. Return the most common normalised
answer only if it is the unique winner and its share meets the threshold. Return None for empty
input, ties or insufficient agreement.


In [ ]:
from collections import Counter

def vote(answers, threshold):
    # TODO: normalise, count, check agreement, and abstain when needed.
    raise NotImplementedError


In [ ]:
assert vote([" Paris", "PARIS", "Lyon"], 2/3) == "paris"
assert vote(["Paris", "Lyon"], 0.5) is None
assert vote(["Paris", "Paris", "Lyon"], 0.8) is None
assert vote([], 0.5) is None
print("OK")


## 5. Explain and measure

**Your answer:** Explain why the tail gets more draws in the higher-temperature setting, why the
longer context exceeds the budget, and why a unanimous vote still needs independent evidence.

**Optional local extension:** If you already have a small instruct model running, repeat one short
prompt at greedy, moderate-temperature nucleus, and high-temperature full-distribution settings.
Keep the model and generated-token limit fixed, save three outputs per setting, and record the
runtime, hardware and decode throughput. Compute throughput as generated tokens divided by decode
seconds, excluding prefill. If you lack a local runtime, report the simulated results and label
them as such.
